# OpTC Host ML — XGB, shared full-data shards

This notebook reads the same persisted 75/10/15 split and vocabulary as the other models.
Every event is retained. Run each model in a separate notebook. No local training or automatic sampling.

Set MANIFEST and VOLUME_ROOTS after attaching the output. Each volume may be mounted under a different Kaggle dataset directory.
Upload/quota decisions depend on the measured final size. Dataset capacity is not notebook RAM.


External-memory XGBoost requires cache capacity and working memory; it does not guarantee that a 12-hour Kaggle session can complete billions of rows.

In [ ]:
from pathlib import Path
import sys, json
MANIFEST = Path('/kaggle/input/optc-host-input/manifest.json')
# Keys must match manifest volumes; point each to its copied volume root.
VOLUME_ROOTS = {
    'ssd': '/kaggle/input/optc-host-input/volumes/ssd',
    'hdd': '/kaggle/input/optc-host-hdd',
    'system': '/kaggle/input/optc-host-system',
}
sys.path.insert(0, str(MANIFEST.parent))
from host_shards import HostShardDataset
from shard_metrics import evaluate_stream
data = HostShardDataset(MANIFEST, VOLUME_ROOTS)
print(json.dumps(data.meta['splits'], indent=2))
print('Full train CSR lower bound GiB:', data.csr_bytes('train') / 2**30)
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)


In [ ]:
MODEL_NAME = 'xgb'
import xgboost as xgb
from host_shards import xgboost_iterator
if not hasattr(xgb, 'ExtMemQuantileDMatrix'):
    raise RuntimeError('This notebook requires XGBoost >= 3.0 for external memory.')
# Configure a writable scratch directory with enough free capacity for XGBoost
# caches. The compressed input size is NOT a bound on cache/training memory.
CACHE = Path('/kaggle/temp/optc_xgb_cache')
CACHE.mkdir(parents=True, exist_ok=True)
train_it = xgboost_iterator(data, 'train', CACHE / 'train')
val_it = xgboost_iterator(data, 'val', CACHE / 'val')
train = xgb.ExtMemQuantileDMatrix(train_it, max_bin=16)
val = xgb.ExtMemQuantileDMatrix(val_it, max_bin=16, ref=train)
model = xgb.train({'objective':'binary:logistic', 'eval_metric':'aucpr',
    'tree_method':'hist', 'device':'cpu', 'max_bin':16, 'max_depth':8,
    'eta':0.05, 'seed':1, 'nthread':4}, train, num_boost_round=300,
    evals=[(val,'validation')], early_stopping_rounds=30)
# Keep only the validation-selected trees for test prediction.
model = model[:model.best_iteration + 1]
predict = lambda X: model.inplace_predict(X)
model.save_model(str(OUTPUT / 'xgb_model.ubj'))


In [ ]:
validation = evaluate_stream(data, 'val', predict)
test = evaluate_stream(data, 'test', predict, threshold=validation['threshold'])
report = {'model': MODEL_NAME, 'split_sha256':data.meta['split_sha256'],
    'vocabulary_sha256':data.meta['vocabulary_sha256'], 'seed':1,
    'validation':validation, 'test':test}
(OUTPUT / (MODEL_NAME + '_metrics.json')).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
